# 🔐 Multi-Agent SOC Architecture with Azure AI Foundry

**Goal:** Build a multi-agent incident triage system using Azure AI Foundry's **Responses API** — the Python-orchestrated Router-Worker pattern.

```
Alert In ──► ORCHESTRATOR (SOC Triage Lead)
                 │
       ┌─────────┴──────────┐
       ▼                    ▼
 Threat Intel Agent   Alert Enrichment Agent
 (IOC lookup,         (MITRE ATT&CK mapping,
  TI feed queries)     playbook retrieval)
       │                    │
       └─────────┬──────────┘
                 ▼
          Triage Report
```

**Pattern:** Python-orchestrated multi-agent — the notebook code calls each worker agent with `responses.create()`, then feeds their outputs into the orchestrator for synthesis. Each agent uses `FunctionTool` with explicit JSON Schema (no more docstring-based auto-parsing).

**SDK:** `azure-ai-projects>=2.0.0` (Responses API, NOT the Assistants API threads/runs model)


In [ ]:
!uv pip install "azure-ai-projects>=2.0.0" azure-identity python-dotenv rich matplotlib pandas -q


In [ ]:
import os
import json
import datetime
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

load_dotenv(Path('..') / '.env', override=False)

AZURE_AI_PROJECT_ENDPOINT = os.getenv('AZURE_AI_PROJECT_ENDPOINT', '')
MODEL_DEPLOYMENT_NAME = os.getenv('MODEL_DEPLOYMENT_NAME', 'gpt-4o')

MOCK_MODE = not bool(AZURE_AI_PROJECT_ENDPOINT)
if MOCK_MODE:
    print('⚠️  MOCK MODE: No AZURE_AI_PROJECT_ENDPOINT found.')
    print('   Cells marked 🔵 run with mock data. Cells marked 🔴 need Azure credentials.')
else:
    print(f'✅ Azure connected: {AZURE_AI_PROJECT_ENDPOINT[:60]}...')
    print(f'   Model: {MODEL_DEPLOYMENT_NAME}')


## 🔵 Visualize the Architecture

Before wiring up Azure, let's visually understand what we're building.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')

agents = [
    dict(x=5.5, y=5.5, w=3, h=1.2, color='#ff6b6b', label='ORCHESTRATOR\n(SOC Triage Lead)', sub='Synthesizes Triage Report'),
    dict(x=1.5, y=2.5, w=3, h=1.2, color='#4ecdc4', label='Threat Intel\nAgent', sub='IOC lookup · TI feeds'),
    dict(x=9.5, y=2.5, w=3, h=1.2, color='#45b7d1', label='Alert Enrichment\nAgent', sub='MITRE · Playbooks'),
]

for a in agents:
    rect = mpatches.FancyBboxPatch((a['x'], a['y']), a['w'], a['h'],
                                   boxstyle='round,pad=0.1', linewidth=2,
                                   edgecolor=a['color'], facecolor=a['color'] + '33')
    ax.add_patch(rect)
    ax.text(a['x'] + a['w']/2, a['y'] + a['h']/2 + 0.15, a['label'],
            ha='center', va='center', fontsize=11, fontweight='bold', color=a['color'], fontfamily='monospace')
    ax.text(a['x'] + a['w']/2, a['y'] + 0.15, a['sub'],
            ha='center', va='center', fontsize=8, color='#8b949e')

# Arrows: Orchestrator ← workers (Python-orchestrated: results fed into orchestrator)
ax.annotate('', xy=(3.0, 3.7), xytext=(6.0, 5.5), arrowprops=dict(arrowstyle='<->', color='#4ecdc4', lw=1.5))
ax.annotate('', xy=(11.0, 3.7), xytext=(8.0, 5.5), arrowprops=dict(arrowstyle='<->', color='#45b7d1', lw=1.5))

# Alert input arrow
ax.annotate('', xy=(7.0, 6.7), xytext=(7.0, 7.2), arrowprops=dict(arrowstyle='->', color='#ff6b6b', lw=2))
ax.text(7.0, 7.3, '🚨 Alert In', ha='center', fontsize=10, color='#ff6b6b', fontweight='bold')

# Output arrow
ax.annotate('', xy=(7.0, 1.8), xytext=(7.0, 2.5), arrowprops=dict(arrowstyle='<-', color='#98d9a8', lw=2))
ax.text(7.0, 1.5, '📄 Triage Report', ha='center', fontsize=10, color='#98d9a8', fontweight='bold')

# Python-orchestrated labels
ax.text(4.3, 4.9, 'responses.create()', fontsize=7.5, color='#4ecdc4', style='italic', rotation=30)
ax.text(8.7, 4.9, 'responses.create()', fontsize=7.5, color='#45b7d1', style='italic', rotation=-30)

ax.set_title('Multi-Agent SOC Architecture — Python-Orchestrated (azure-ai-projects 2.x Responses API)',
             color='white', fontsize=12, pad=15)
plt.tight_layout()
plt.show()


## 🔵 Load Sample Alerts

Real SOC alerts from our `data/sample_alerts.json` — these will be triaged by our multi-agent system.

In [ ]:
alerts_path = Path('../notebooks/data/sample_alerts.json')
if not alerts_path.exists():
    alerts_path = Path('data/sample_alerts.json')  # if running from notebooks/

with open(alerts_path) as f:
    alerts = json.load(f)

df_alerts = pd.DataFrame(alerts)

# Severity color mapping
severity_colors = {'Critical': '#ff4444', 'High': '#ff9944', 'Medium': '#ffdd44', 'Low': '#44bbff'}

display(df_alerts[['alert_id', 'timestamp', 'severity', 'source', 'title']]
        .style.map(lambda v: f'color: {severity_colors.get(v, "white")}', subset=['severity'])
        .set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6'}))

In [ ]:
# Visualize alert distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#1e2127')

sev_counts = df_alerts['severity'].value_counts()
colors = [severity_colors.get(s, '#888') for s in sev_counts.index]
ax1.barh(sev_counts.index, sev_counts.values, color=colors, edgecolor='#444', height=0.6)
ax1.set_facecolor('#1e2127')
ax1.set_title('Alerts by Severity', color='white')
ax1.tick_params(colors='white')
ax1.spines[:].set_color('#444')
for spine in ax1.spines.values():
    spine.set_color('#444')

src_counts = df_alerts['source'].value_counts()
ax2.pie(src_counts.values, labels=src_counts.index, autopct='%1.0f%%',
        colors=['#4ecdc4', '#45b7d1', '#ff6b6b'], textprops={'color': 'white'})
ax2.set_facecolor('#1e2127')
ax2.set_title('Alerts by Source', color='white')

plt.suptitle('SOC Alert Landscape — Today', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 🔴 Connect to Azure AI Foundry

Set `AZURE_AI_PROJECT_ENDPOINT` and `MODEL_DEPLOYMENT_NAME` in your `.env` file (see `.env.example`).

The new SDK uses **`azure-ai-projects>=2.0.0`** which provides:
- `AIProjectClient` — manages agent lifecycle (`create_version`, `delete_version`)
- `project_client.get_openai_client()` — returns an OpenAI client bound to your Foundry project
- `openai_client.responses.create(...)` — the Responses API (stateless or conversation-scoped)
- `openai_client.conversations.*` — multi-turn conversation management


In [ ]:
if not MOCK_MODE:
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    project_client = AIProjectClient(
        endpoint=AZURE_AI_PROJECT_ENDPOINT,
        credential=credential,
    )
    openai_client = project_client.get_openai_client()
    print('✅ AIProjectClient + openai_client ready')
else:
    print('⚠️  Skipping — set AZURE_AI_PROJECT_ENDPOINT to connect to Azure')


## 🔴 Step 1 — Create Specialist Agents

Each specialist agent has a focused responsibility and a matching set of **FunctionTools**.
These tools simulate querying a real SIEM/threat intel feed — swap in real API calls for production.

In [ ]:
# ── Tool functions for the Threat Intel Agent ──────────────────────────────
import json as _json

def lookup_ioc_threat_intel(indicator: str) -> str:
    """Look up an IOC in threat intelligence feeds."""
    # Production: call VirusTotal, MSFT TI, or Sentinel TAXII feed
    mock_db = {
        '185.220.101.47': {'reputation': 'malicious', 'type': 'TOR exit node', 'confidence': 0.97, 'actor': 'Unknown/TOR', 'first_seen': '2022-03-01'},
        '194.165.16.101': {'reputation': 'malicious', 'type': 'C2 server', 'confidence': 0.95, 'actor': 'APT29', 'first_seen': '2024-11-12'},
    }
    result = mock_db.get(indicator, {'reputation': 'unknown', 'confidence': 0.0, 'type': 'no data'})
    return _json.dumps({'indicator': indicator, **result})


def query_siem_related_alerts(alert_id: str, lookback_hours: int = 48) -> str:
    """Query the SIEM for related alerts correlated with the given alert."""
    mock_correlations = {
        'ALT-2025-001': [{'id': 'ALT-2025-005', 'reason': 'same user account', 'score': 0.82}],
        'ALT-2025-002': [{'id': 'ALT-2025-002', 'reason': 'same host', 'score': 0.91}],
    }
    found = mock_correlations.get(alert_id, [])
    return _json.dumps({'alert_id': alert_id, 'lookback_hours': lookback_hours, 'correlated_alerts': found})


# ── FunctionTool schemas for azure-ai-projects 2.x (explicit JSON Schema) ───
# No more docstring-based auto-parsing — each tool needs name, description, parameters, strict=True
from azure.ai.projects.models import FunctionTool

ti_tool_lookup = FunctionTool(
    name='lookup_ioc_threat_intel',
    description='Look up an indicator of compromise (IP, domain, file hash) in threat intelligence feeds. Returns reputation, actor attribution, and confidence score.',
    parameters={
        'type': 'object',
        'properties': {
            'indicator': {'type': 'string', 'description': 'IOC to look up — IPv4 address, domain name, or MD5/SHA256 hash.'},
        },
        'required': ['indicator'],
        'additionalProperties': False,
    },
    strict=True,
)

ti_tool_correlate = FunctionTool(
    name='query_siem_related_alerts',
    description='Query SIEM for alerts correlated with the given alert ID. Returns correlated alert IDs and correlation scores.',
    parameters={
        'type': 'object',
        'properties': {
            'alert_id':       {'type': 'string',  'description': 'Alert ID to find correlations for.'},
            'lookback_hours': {'type': 'integer', 'description': 'Hours to look back in SIEM. Default 48.'},
        },
        'required': ['alert_id'],
        'additionalProperties': False,
    },
    strict=True,
)

TI_FUNCTIONS = {
    'lookup_ioc_threat_intel': lookup_ioc_threat_intel,
    'query_siem_related_alerts': query_siem_related_alerts,
}

print('✅ Threat Intel tool functions + FunctionTool schemas defined')


In [ ]:
# ── Tool functions for the Alert Enrichment Agent ─────────────────────────

def map_to_mitre_attack(alert_title: str, description: str) -> str:
    """Map an alert to MITRE ATT&CK tactics and techniques."""
    # Production: call MITRE ATT&CK TAXII feed or Azure Sentinel entity mapping API
    keywords = (alert_title + ' ' + description).lower()
    mappings = []
    if 'credential' in keywords or 'stuffing' in keywords or 'login' in keywords:
        mappings.append({'tactic': 'Initial Access', 'technique': 'T1078', 'name': 'Valid Accounts'})
        mappings.append({'tactic': 'Credential Access', 'technique': 'T1110.004', 'name': 'Credential Stuffing'})
    if 'cobalt strike' in keywords or 'beacon' in keywords or 'c2' in keywords:
        mappings.append({'tactic': 'Command and Control', 'technique': 'T1071.001', 'name': 'Web Protocols'})
        mappings.append({'tactic': 'Defense Evasion', 'technique': 'T1055.012', 'name': 'Process Hollowing'})
    if 'powershell' in keywords or 'encoded' in keywords:
        mappings.append({'tactic': 'Execution', 'technique': 'T1059.001', 'name': 'PowerShell'})
        mappings.append({'tactic': 'Defense Evasion', 'technique': 'T1027', 'name': 'Obfuscated Files'})
    if not mappings:
        mappings.append({'tactic': 'Unknown', 'technique': 'TBD', 'name': 'Requires manual review'})
    return _json.dumps({'alert_title': alert_title, 'mitre_mappings': mappings})


def get_response_playbook(attack_technique_id: str) -> str:
    """Retrieve the incident response playbook for a given MITRE ATT&CK technique ID."""
    playbooks = {
        'T1078': {'priority': 'P1', 'steps': ['Revoke user sessions immediately', 'Force MFA re-registration', 'Block source IP', 'Export 48h sign-in logs', 'Check for persistence mechanisms']},
        'T1110.004': {'priority': 'P2', 'steps': ['Block originating IP range', 'Enable smart lockout if not active', 'Review authentication logs for success after failures', 'Notify affected users']},
        'T1059.001': {'priority': 'P1', 'steps': ['Isolate affected endpoint', 'Capture memory dump', 'Decode base64 payload', 'Check for scheduled tasks/run keys', 'Hunt for lateral movement']},
        'T1055.012': {'priority': 'P1', 'steps': ['Isolate host immediately', 'Capture network traffic (pcap)', 'Identify parent process chain', 'Block C2 IP at perimeter', 'Submit sample to sandbox']},
    }
    result = playbooks.get(attack_technique_id.split('.')[0], {'priority': 'P3', 'steps': ['Investigate manually', 'Escalate to Tier 2']})
    return _json.dumps({'technique': attack_technique_id, **result})


# ── FunctionTool schemas for the enrichment agent ──────────────────────────
enrich_tool_mitre = FunctionTool(
    name='map_to_mitre_attack',
    description='Map a security alert to MITRE ATT&CK tactics and techniques using the alert title and description.',
    parameters={
        'type': 'object',
        'properties': {
            'alert_title': {'type': 'string', 'description': 'Title of the security alert.'},
            'description': {'type': 'string', 'description': 'Detailed description of the security alert.'},
        },
        'required': ['alert_title', 'description'],
        'additionalProperties': False,
    },
    strict=True,
)

enrich_tool_playbook = FunctionTool(
    name='get_response_playbook',
    description='Retrieve incident response playbook steps for a MITRE ATT&CK technique ID (e.g., T1078 or T1059.001). Returns priority and ordered response steps.',
    parameters={
        'type': 'object',
        'properties': {
            'attack_technique_id': {'type': 'string', 'description': 'MITRE ATT&CK technique ID like T1078 or T1059.001.'},
        },
        'required': ['attack_technique_id'],
        'additionalProperties': False,
    },
    strict=True,
)

ENRICH_FUNCTIONS = {
    'map_to_mitre_attack': map_to_mitre_attack,
    'get_response_playbook': get_response_playbook,
}

print('✅ Alert Enrichment tool functions + FunctionTool schemas defined')


In [ ]:
if not MOCK_MODE:
    from azure.ai.projects.models import PromptAgentDefinition

    # ── Threat Intel Agent ──────────────────────────────────────────────────
    threat_intel_agent = project_client.agents.create_version(
        agent_name='soc-threat-intel-agent',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are a SOC Threat Intelligence Analyst. Investigate indicators of compromise '
                '(IOCs) and correlate alerts with known threat actors. Always call the available '
                'tools to look up IOCs and report on actor attribution, confidence scores, and '
                'correlated alerts. Return your analysis as structured JSON.'
            ),
            tools=[ti_tool_lookup, ti_tool_correlate],
        ),
    )
    print(f'✅ Threat Intel Agent: {threat_intel_agent.name} v{threat_intel_agent.version}')

    # ── Alert Enrichment Agent ──────────────────────────────────────────────
    alert_enrichment_agent = project_client.agents.create_version(
        agent_name='soc-alert-enrichment-agent',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are a SOC Alert Enrichment Specialist. Map security alerts to MITRE ATT&CK '
                'tactics and techniques using map_to_mitre_attack, then retrieve the incident '
                'response playbook using get_response_playbook. Return structured JSON with '
                'technique IDs, tactic names, and prioritized response steps.'
            ),
            tools=[enrich_tool_mitre, enrich_tool_playbook],
        ),
    )
    print(f'✅ Alert Enrichment Agent: {alert_enrichment_agent.name} v{alert_enrichment_agent.version}')

else:
    class MockAgent:
        def __init__(self, name):
            self.name = name
            self.version = 'v1'
    threat_intel_agent = MockAgent('soc-threat-intel-agent')
    alert_enrichment_agent = MockAgent('soc-alert-enrichment-agent')
    print('⚠️  Using mock agents — set AZURE_AI_PROJECT_ENDPOINT to create real agents')


## 🔴 Step 2 — Create the Orchestrator

The orchestrator synthesizes findings from both specialist agents. In this **Python-orchestrated** pattern:

1. The notebook calls each specialist agent using `responses.create()`
2. Each call may trigger `FunctionTool` invocations (resolved via a function-call loop)
3. The orchestrator receives both analyses as input and produces the final Triage Report

> 💡 This pattern is fully transparent: you can log, inspect, and debug every sub-agent response locally. For server-side routing with registered A2A connections, see `A2APreviewTool` (requires configuring an A2A project connection in Azure AI Foundry).


In [ ]:
if not MOCK_MODE:
    orchestrator = project_client.agents.create_version(
        agent_name='soc-orchestrator',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are the SOC Triage Lead — the master incident orchestrator. '
                'Given analyses from specialist agents, synthesize a structured Triage Report with:\n'
                '  - Executive Summary (2-3 sentences)\n'
                '  - IOC Analysis (from threat intel findings)\n'
                '  - MITRE Mapping + Response Steps (from enrichment findings)\n'
                '  - Severity Assessment and Priority\n'
                '  - Recommended Next Actions\n'
                'Be precise, actionable, and concise. This report goes directly to the on-call analyst.'
            ),
        ),
    )
    print(f'✅ Orchestrator created: {orchestrator.name} v{orchestrator.version}')
    print('   Python-orchestrated pattern: worker agents invoked explicitly from notebook code')
else:
    class MockAgent:
        def __init__(self, name):
            self.name = name
            self.version = 'v1'
    orchestrator = MockAgent('soc-orchestrator')
    print('⚠️  Skipping orchestrator creation — set AZURE_AI_PROJECT_ENDPOINT')


## 🔴 Step 3 — Investigate an Alert!

We'll send Alert `ALT-2025-002` (Cobalt Strike Beacon) to the orchestrator and watch the agents collaborate.

In [ ]:
# Pick the critical alert
alert = next(a for a in alerts if a['alert_id'] == 'ALT-2025-002')
print(json.dumps(alert, indent=2))

In [ ]:
if not MOCK_MODE:
    from openai.types.responses.response_input_param import FunctionCallOutput, ResponseInputParam

    def run_agent_turn(agent, user_input: str, fn_map: dict | None = None) -> str:
        """Single agent turn with automatic function call resolution (Responses API)."""
        AGENT_REF = {'agent_reference': {'name': agent.name, 'type': 'agent_reference'}}
        response = openai_client.responses.create(
            input=user_input,
            extra_body=AGENT_REF,
        )
        for _ in range(8):  # max function-call resolution rounds
            calls = [item for item in response.output if item.type == 'function_call']
            if not calls:
                break
            inputs: ResponseInputParam = [
                FunctionCallOutput(
                    type='function_call_output',
                    call_id=fc.call_id,
                    output=(
                        _json.dumps((fn_map or {})[fc.name](**_json.loads(fc.arguments)))
                        if fn_map and fc.name in fn_map
                        else '{"error": "unknown function"}'
                    ),
                )
                for fc in calls
            ]
            response = openai_client.responses.create(
                input=inputs,
                previous_response_id=response.id,
                extra_body=AGENT_REF,
            )
        return response.output_text

    alert_brief = (
        f'Alert ID: {alert["alert_id"]} | Severity: {alert["severity"]}\n'
        f'Title: {alert["title"]}\n'
        f'Description: {alert["description"]}\n'
        f'Source IP: {alert["source_ip"]} | IOCs: {", ".join(alert["indicators"])}\n'
        f'Affected users: {", ".join(alert["affected_users"])}'
    )

    print('🔍 Step 1/3: Threat Intel Agent investigating IOCs...')
    ti_result = run_agent_turn(threat_intel_agent, f'Investigate all IOCs in this alert:\n{alert_brief}', TI_FUNCTIONS)
    print(f'   ✅ TI analysis complete ({len(ti_result)} chars)')

    print('🔍 Step 2/3: Alert Enrichment Agent mapping to MITRE...')
    enrich_result = run_agent_turn(alert_enrichment_agent, f'Map this alert to MITRE ATT&CK and retrieve playbook:\n{alert_brief}', ENRICH_FUNCTIONS)
    print(f'   ✅ Enrichment complete ({len(enrich_result)} chars)')

    # Orchestrator uses a conversation so we can resume in Notebook 03
    print('🔍 Step 3/3: Orchestrator synthesizing Triage Report...')
    triage_conversation = openai_client.conversations.create()
    triage_response = openai_client.responses.create(
        conversation=triage_conversation.id,
        input=(
            f'Synthesize a Triage Report from these specialist analyses.\n\n'
            f'=== THREAT INTEL ANALYSIS ===\n{ti_result}\n\n'
            f'=== ALERT ENRICHMENT ANALYSIS ===\n{enrich_result}\n\n'
            f'=== ORIGINAL ALERT ===\n{alert_brief}'
        ),
        extra_body={'agent_reference': {'name': orchestrator.name, 'type': 'agent_reference'}},
    )
    print('\n📋 TRIAGE REPORT:')
    print('=' * 60)
    print(triage_response.output_text)
    print(f'\n💾 Conversation ID (save for Notebook 03): {triage_conversation.id}')

else:
    triage_response = None
    triage_conversation = None
    print('📋 EXAMPLE TRIAGE REPORT (mock):')
    print('=' * 60)
    print('''
## SOC Triage Report — ALT-2025-002

### Executive Summary
A Cobalt Strike beacon was detected on endpoint 10.0.0.45, attributed to APT29
with 95% confidence. The C2 IP 194.165.16.101 has been previously flagged.
Immediate containment is required — classify as Priority 1 incident.

### IOC Analysis (Threat Intel Agent)
- 194.165.16.101: MALICIOUS | Type: C2 server | Actor: APT29 | Confidence: 0.95
- First seen: 2024-11-12 | Associated with SolarWinds-style campaign

### MITRE Mapping (Enrichment Agent)
- T1055.012 — Process Hollowing (Defense Evasion)
- T1071.001 — Web Protocols (Command and Control)

### Response Steps (Priority: P1)
1. ISOLATE host 10.0.0.45 immediately
2. CAPTURE pcap from host before isolation
3. BLOCK 194.165.16.101 at perimeter + NSG
4. IDENTIFY parent process chain for explorer.exe PID 4872
5. SUBMIT memory dump to sandbox
6. HUNT for lateral movement from svc-backup account
    ''')


In [ ]:
if not MOCK_MODE:
    agent_calls = [
        {'sub_agent': threat_intel_agent.name,    'output_len': len(ti_result),    'fn_calls': '2 (lookup_ioc, siem_query)'},
        {'sub_agent': alert_enrichment_agent.name, 'output_len': len(enrich_result), 'fn_calls': '2 (mitre_map, playbook)'},
        {'sub_agent': orchestrator.name,           'output_len': len(triage_response.output_text), 'fn_calls': '0 (synthesis only)'},
    ]
    df_calls = pd.DataFrame(agent_calls)
    print('Python-Orchestrated Agent Call Log:')
    display(df_calls)
else:
    agent_calls = [
        {'sub_agent': 'soc-threat-intel-agent',    'output_len': 312, 'fn_calls': '2 (lookup_ioc, siem_query)'},
        {'sub_agent': 'soc-alert-enrichment-agent', 'output_len': 498, 'fn_calls': '2 (mitre_map, playbook)'},
        {'sub_agent': 'soc-orchestrator',           'output_len': 950, 'fn_calls': '0 (synthesis only)'},
    ]
    df_calls = pd.DataFrame(agent_calls)
    print('Simulated Agent Call Log:')
    display(df_calls)


## 🔵 Visualize the Multi-Agent Call Chain

The Python-orchestrated pattern gives you full visibility — each `responses.create()` call is captured locally. Below we visualize the agent call log and output sizes from the triage run above.


In [ ]:
# Visualize the Python-orchestrated multi-agent call chain
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#1e2127')
ax.set_facecolor('#1e2127')

agents_called = [c['sub_agent'] for c in agent_calls]
output_sizes  = [c['output_len'] for c in agent_calls]
agent_colors  = ['#4ecdc4', '#45b7d1', '#ff6b6b']

bars = ax.barh(agents_called, output_sizes, color=agent_colors[:len(agents_called)], height=0.5)
ax.set_xlabel('Response Size (chars)', color='white')
ax.set_title('Python-Orchestrated Multi-Agent Call Chain', color='white', pad=10)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#444')
for bar, size in zip(bars, output_sizes):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, f'{size}', va='center', color='white', fontsize=9)

# Add annotation showing the orchestration flow
ax.text(0.98, 0.05, '① TI → ② Enrich → ③ Orchestrator →  Triage Report',
        transform=ax.transAxes, ha='right', fontsize=8, color='#8b949e', style='italic')

plt.tight_layout()
plt.show()


## 🔴 Cleanup

In [ ]:
if not MOCK_MODE:
    # Delete conversation if one was created
    if triage_conversation:
        openai_client.conversations.delete(conversation_id=triage_conversation.id)

    # Delete all agents using delete_version
    for agent in [orchestrator, alert_enrichment_agent, threat_intel_agent]:
        project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
        print(f'🗑️  Deleted {agent.name}')

    openai_client.close()
    project_client.close()
    print('\n✅ All agents and conversation deleted.')
else:
    print('⚠️  Nothing to clean up in mock mode.')
